<a href="https://colab.research.google.com/github/anthony-yt/deepwater-image-segmentation-classical-quantum/blob/main/notebooks/3_4_final_retraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 4.8 — Reentrenamiento final: la tabla comparativa del Goal 2.4

Este notebook **no implementa el protocolo**: lo ejecuta. Toda la lógica vive en módulos que los
tres modelos comparten, para que las filas de la tabla se puedan concatenar sin traducciones:

| pieza | de quién | para qué |
|---|---|---|
| `src/training/final_retraining.py` | Dayana | manifiesto, presupuesto, materialización del fold, métricas, tabla |
| `src/evaluation/segmentation_metrics.py` | Anthony (Task 5.7) | las cinco métricas y la línea base «todo estructura» |
| `src/training/unet_final.py` | Ariana | el lado del U-Net del mismo contrato |

Las ocho decisiones están en `docs/protocolo_evaluacion.md`, acordadas con Dayana el 2026-11-08.
Lo que hay que recordar al leer la salida:

- **No se elige el mejor checkpoint** (decisión 3). Se entrena el presupuesto acordado en
  *pasadas sobre los datos* y se mide con los pesos finales. La Task 4.2 mostró que mirar el
  test para elegir la época es justo lo que infla el resultado.
- **El manifiesto es un archivo con su `sha256`**, no un acuerdo verbal. Una corrida que no
  registre ese hash no entra en la tabla.
- **Cada columna se lee contra su propia línea base**, que va de 0,081 en Video6 a 0,929 en
  Video2. Un IoU de 0,72 es un empate con no hacer nada en Video1 y un resultado excelente en
  Video6.

## 1. Entorno

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO = Path("/content/deepwater-image-segmentation-classical-quantum")
    if not REPO.exists():
        raise SystemExit("clona el repo en /content y vuelve a ejecutar esta celda")
    DRIVE = Path("/content/drive/MyDrive/Under_Water_Dataset/Dataset_Processed")
else:
    REPO = Path.cwd().parent
    DRIVE = Path(r"G:/Mi unidad/Under_Water_Dataset/Dataset_Processed")

sys.path.insert(0, str(REPO))

print("repo :", REPO)
print("drive:", DRIVE, "| existe:", DRIVE.exists())
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 2. Configuración y preflight

El preflight no entrena: valida el `sha256` del manifiesto, comprueba que los 62 pares
imagen/máscara existen y miden lo mismo, y muestra el plan de corridas con el presupuesto en
iteraciones. Si algo no cuadra, falla **antes** de gastar GPU.

In [ ]:
from src.training.unet_final import execute, preflight

MANIFEST_SHA256 = "ebebc379148f82bb8f914a4e1f8310a5de4acdab9a93c62f3a645db51d88106a"
PASADAS = 20  # valor del piloto de Dayana (resultados_4_8_p20); el final se fija en la decisión 3

SETTINGS = {
    "mode": "pilot",
    "protocol_confirmed": True,
    "models": ["unet"],
    "seeds": [0],
    "folds": ["Video1"],
    "expected_videos": [f"Video{i}" for i in range(1, 7)],
    "expected_counts": {"Video1": 22, **{f"Video{i}": 8 for i in range(2, 7)}},
    "passes": PASADAS,
    "batch_size": 2,
    "learning_rate": 1e-4,
    "loss": "ce_dice",
    "attention": False,
    "scheduler": "cosine",
    "augment": False,
    "img_size": 512,
    "manifest": str(DRIVE / "Seleccion" / "folds_4_8.csv"),
    "manifest_sha256": MANIFEST_SHA256,
    "frames_dir": str(DRIVE / "Frames_CLAHE"),
    "masks_dir": str(DRIVE / "Test_Set_Curado"),
    "results_dir": str(DRIVE / "resultados_4_8_p20"),
}

plan = execute(SETTINGS, run=False)

### 2.1 El conjunto que fija el manifiesto

`folds_4_8.meta.json` es la fuente de la línea base por video. Se lee aquí porque es lo que da
sentido a la tabla final, y porque el manifiesto deja fuera las 8 máscaras de Video7: entraron el
2026-11-10, después de que el manifiesto se generara, y meterlas cambiaría el `sha256` que ya
registraron las corridas de Dayana.

In [ ]:
META = json.loads((DRIVE / "Seleccion" / "folds_4_8.meta.json").read_text(encoding="utf-8"))
assert META["sha256"] == MANIFEST_SHA256, "el meta no describe el manifiesto que vamos a usar"

base = pd.DataFrame(META["por_video"]).T.rename(columns={
    "n_mascaras": "mascaras", "peso_en_el_conjunto": "peso",
    "base_iou_todo_estructura": "linea_base_iou"})
print(base.to_string(), "\n")
print("fuera del manifiesto:", META["mascaras_en_la_carpeta_fuera_del_manifiesto"])
print("margen sobre la línea base = 1 - base:")
print((1 - base.linea_base_iou.astype(float)).round(4).to_string())

## 3. Piloto: un fold, una semilla

Sirve para dos cosas: comprobar que el presupuesto de 20 pasadas alcanza para que el U-Net salga
de la predicción constante, y dejar una corrida comparable con las dos de Dayana en
`resultados_4_8_p20/pilot` (SegFormer 0,719 y UWSegFormer 0,865 de IoU en Video1, contra una línea
base de 0,741).

La corrida es reanudable: cada carpeta guarda `completed.json` con el hash de su configuración, y
una segunda ejecución la reutiliza en vez de repetirla.

In [ ]:
PILOTO = True

if PILOTO:
    resumen_piloto = execute(SETTINGS, run=True)
    display(resumen_piloto)
else:
    print("piloto desactivado")

## 4. Corridas finales: 6 folds × 5 semillas

30 corridas del U-Net. `mode="final"` exige protocolo confirmado, de 3 a 5 semillas y **todos**
los folds, y además exporta un `per_frame_unet_<fold>_seed<n>.csv` a la carpeta compartida, que es
de donde la tabla de los tres modelos lee.

`results_dir` tiene que ser la misma carpeta que use Dayana para sus corridas finales.

In [ ]:
RESULTADOS_FINALES = DRIVE / "resultados_4_8_final"

FINAL = {**SETTINGS, "mode": "final", "seeds": [0, 1, 2, 3, 4], "folds": None,
         "results_dir": str(RESULTADOS_FINALES)}

CORRER_FINAL = False  # ponlo en True cuando el piloto haya fijado las pasadas

if CORRER_FINAL:
    resumen_final = execute(FINAL, run=True)
    display(resumen_final)
else:
    execute(FINAL, run=False)

## 5. La tabla de los tres modelos

La arma `aggregate_shared_results` de Dayana, que se niega a publicar una tabla parcial: exige los
tres modelos, los mismos folds y semillas, cobertura exacta de cada test y un único
`manifest_sha256` en todas las filas.

In [ ]:
from src.training.final_retraining import aggregate_shared_results

SETTINGS_SEGFORMERS = {
    **{k: FINAL[k] for k in ("mode", "protocol_confirmed", "seeds", "folds", "expected_videos",
                             "expected_counts", "passes", "batch_size", "learning_rate",
                             "manifest", "manifest_sha256", "frames_dir", "masks_dir",
                             "results_dir")},
    "models": ["segformer", "uwsegformer"],
    "freeze_backbone": True,
    "uws_root": "/content/UWSegFormer",
    "checkpoints": {"segformer": "/content/segformer_mit_b0_ade20k.pth",
                    "uwsegformer": str(DRIVE.parent / "checkpoints" / "uws_SUIM.pth")},
}

try:
    tabla = aggregate_shared_results(SETTINGS_SEGFORMERS)
    display(tabla)
except Exception as exc:
    tabla = None
    print("todavía no hay tabla:", type(exc).__name__, exc)

## 6. Cada columna contra su línea base

La tabla sola no se puede leer. Esta celda le pega la línea base de cada video y la diferencia
contra ella, que es el número que de verdad dice si el modelo aprendió algo.

In [ ]:
def contra_linea_base(tabla: pd.DataFrame) -> pd.DataFrame:
    """Añade la línea base por video y la diferencia del IoU medio contra ella."""
    t = tabla[tabla.video != "GLOBAL_macro_videos"].copy()
    t["linea_base"] = t.video.map(base.linea_base_iou.astype(float))
    t["diff_vs_base"] = t.iou_mean - t.linea_base
    t["margen_disponible"] = 1 - t.linea_base
    t["aprovechado"] = t.diff_vs_base / t.margen_disponible
    columnas = ["modelo", "video", "iou_mean", "iou_std", "linea_base", "diff_vs_base",
                "margen_disponible", "aprovechado", "n_seeds"]
    return t[columnas].sort_values(["video", "modelo"]).round(4)


if tabla is not None:
    display(contra_linea_base(tabla))

## 7. La regla de empate (decisión 4)

Dos modelos se declaran distintos solo si `|media_A − media_B| > sd_A + sd_B`, con la SD entre
semillas. El bootstrap es el desempate y el arnés todavía no da el intervalo emparejado entre
modelos: es el hueco conocido de la decisión 8.

In [ ]:
def empate(media_a, sd_a, media_b, sd_b) -> bool:
    """True si la diferencia no supera la suma de las desviaciones entre semillas."""
    if any(pd.isna(v) for v in (media_a, sd_a, media_b, sd_b)):
        raise ValueError("falta la SD: hacen falta al menos 2 semillas por celda")
    return abs(media_a - media_b) <= sd_a + sd_b


def comparar(tabla: pd.DataFrame, metrica: str = "iou") -> pd.DataFrame:
    filas = []
    for video, g in tabla.groupby("video"):
        g = g.set_index("modelo")
        for a, b in [("U-Net", "SegFormer"), ("U-Net", "UWSegFormer"),
                     ("SegFormer", "UWSegFormer")]:
            if a not in g.index or b not in g.index:
                continue
            ma, sa = g.loc[a, f"{metrica}_mean"], g.loc[a, f"{metrica}_std"]
            mb, sb = g.loc[b, f"{metrica}_mean"], g.loc[b, f"{metrica}_std"]
            filas.append({"video": video, "A": a, "B": b, "media_A": ma, "media_B": mb,
                          "diferencia": ma - mb, "sd_A+sd_B": sa + sb,
                          "veredicto": "empate" if empate(ma, sa, mb, sb) else
                                       f"gana {a if ma > mb else b}"})
    return pd.DataFrame(filas).round(4)


if tabla is not None:
    display(comparar(tabla))

## 8. Conclusión

Se rellena con la tabla delante, no antes. Lo que ya se sabe del piloto de Dayana en Video1
(línea base 0,741):

| modelo | pasadas | IoU | contra la línea base |
|---|---|---|---|
| SegFormer | 1 | 0,268 | −0,473 |
| UWSegFormer | 1 | 0,683 | −0,058 |
| SegFormer | 20 | 0,719 | −0,021 |
| UWSegFormer | 20 | **0,865** | **+0,124** |

Es decir: con una sola pasada ningún modelo le gana a marcar todo el frame, y con 20 solo
UWSegFormer lo hace. Eso ya responde en parte al Goal 2.4, y marca el presupuesto como la variable
que más mueve la tabla.

Lo que falta para cerrar la task: las 30 corridas del U-Net, las 30 de cada SegFormer, y el
intervalo bootstrap emparejado entre modelos para aplicar la regla de empate con algo más que la
SD entre semillas.